In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error
np.random.seed(13);tf.random.set_seed(13)
# Synthetic hourly electricity demand for one year, with daily/weekly/seasonal patterns
dates=pd.date_range("2024-01-01",periods=365*24,freq="h");t=np.arange(len(dates))
demand=180+25*np.sin(2*np.pi*t/24-1)+12*np.sin(2*np.pi*t/(24*7))+18*np.sin(2*np.pi*t/(24*365))+np.random.normal(0,4,len(t))
series=pd.Series(demand,index=dates);series.iloc[[50,51,800]]=np.nan;series=series.interpolate()
scaler=MinMaxScaler();scaled=scaler.fit_transform(series.values.reshape(-1,1)).ravel()
window=24
X=np.array([scaled[i-window:i] for i in range(window,len(scaled))]);y=scaled[window:]
split=int(.8*len(X));X_train,X_test=X[:split],X[split:];y_train,y_test=y[:split],y[split:]
X_train=X_train[...,None];X_test=X_test[...,None]
model=tf.keras.Sequential([tf.keras.layers.Input((window,1)),tf.keras.layers.SimpleRNN(32),tf.keras.layers.Dense(1)])
model.compile(optimizer="adam",loss="mse");h=model.fit(X_train,y_train,validation_split=.1,epochs=15,batch_size=64,verbose=1)
pred=scaler.inverse_transform(model.predict(X_test,verbose=0));actual=scaler.inverse_transform(y_test.reshape(-1,1))
print("MAE:",round(mean_absolute_error(actual,pred),3));print("RMSE:",round(np.sqrt(mean_squared_error(actual,pred)),3))
plt.figure(figsize=(12,4));plt.plot(actual[:300],label="Actual");plt.plot(pred[:300],label="Predicted");plt.title("Actual vs predicted hourly electricity demand");plt.xlabel("Test hour");plt.ylabel("Demand");plt.legend();plt.grid();plt.show()
plt.plot(h.history["loss"],label="Train");plt.plot(h.history["val_loss"],label="Validation");plt.title("SimpleRNN training/validation loss");plt.xlabel("Epoch");plt.ylabel("MSE");plt.legend();plt.grid();plt.show()